# Compartment - gauss-only pipeline (Kaggle) - train80

Chạy **train80** (compound-level 80/20 split) với package gọn `src/`: chỉ gauss head
(`N(mu, sigma^2)` per role), không reg/softmax heads, không MLM warmup.

### Lấy repo
- Notebook clone `REPO_URL` (mặc định `AmnO-O/MoTune`, nhánh `main`) vào `/kaggle/working/Compartment`.
- Repo **private**: đặt `GITHUB_TOKEN` (PAT) dưới dạng `os.environ['GITHUB_TOKEN']`.
- Data vẫn đọc từ Kaggle mount (`/kaggle/input/datasets/ieltsmater/compartment/Compartment`).

### Knobs
- `MODE = 'all' | 'train80'` (all == train80).
- `OVERRIDES`: chỉnh hyperparams + gauss context knobs (`gauss_dedicated`,
  `gauss_ctx_mod/head/pv`, `context_layers`, `span_layers`, ...). Giá trị trống `''` = giữ default.
- `TRAIN_EXTRA`: thêm `--set key=value` cách nhau bằng dấu phẩy.

Outputs ở `/kaggle/working`: `models/best.pt`, `metrics.json` (rho Mod/Head/Mean trên val 20%),
`history.json`, `config.json`. predict/train5/resume sẽ wire sau.

In [ ]:
import os, subprocess, sys
from pathlib import Path

def _importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

# ---- repo source -----------------------------------------------------------
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/AmnO-O/MoTune.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')    # set for private repos

dest = Path('/kaggle/working/Compartment')
url = REPO_URL
if GITHUB_TOKEN:
    url = url.replace('https://', f'https://{GITHUB_TOKEN}@')
dest.parent.mkdir(parents=True, exist_ok=True)
if (dest / 'src' / 'run.py').is_file():
    print('refreshing existing clone at', dest)
    subprocess.run(['git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(dest), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    print('cloning', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, url, str(dest)], check=True)
REPO = dest
os.chdir(REPO)
print('repo:', REPO)
print('head:', subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# ---- config file (optional) ------------------------------------------------
CONFIG_FILE = os.environ.get('CONFIG_FILE', '')   # e.g. 'config/mae_gauss.json'

# ---- knobs ----------------------------------------------------------------
MODE = os.environ.get('MODE', 'all')      # all | train80
if MODE not in ('all', 'train80'):
    MODE = 'train80'
TRAIN_EXTRA = os.environ.get('TRAIN_EXTRA', '')   # extra --set flags, comma-separated

# ---- tunable config overrides ('' = keep src.py default) ------------------
# Edit values here to tune a run without touching --set flags below.
OVERRIDES = {
    'freeze_epochs': '',
    'lora_epochs': '',
    'lora_rank': '',
    'lora_alpha': '',
    'lora_from_layer': '18',                # LoRA window: top layers (0 = all)
    'batch_size': '',
    'head_lr': '',
    'encoder_lr': '',
    'ccc_weight': '',
    'lambda_rank': '',
    'lambda_compound': '',
    'lambda_dist': '',                      # weight of KL(N(mu_p,sig_p)||N(y,sig_t))
    'bin_sigma': '',
    'num_workers': '',
    # --- pv lineage toggles ('' = config default: en on, de off) ---
    'en_pv_train': '',                       # e.g. 'en-pv-train.tsv' (en PV added to train80 by default)
    'de_pv_train': '',                       # German PV ~<5% alignable in this corpus; set to add as representation-only rows
    'aux_data_paths': '',                     # e.g. 'nctti_en.tsv' label-free rows
    # --- gauss feature / context knobs ---
    'context_layers': '',                     # e.g. 10,16,22 = mean-pool 3 global layers ('' = last)
    'span_layers': '',                        # e.g. -1 = last layer ('' = auto mid-5)
    'gauss_dedicated': '',                     # '1' = mod/head gauss heads read context at their OWN layer
    'gauss_ctx_mod': '',                        # context layer for modifier head (e.g. 19 = block 18 global)
    'gauss_ctx_head': '',                       # context layer for head-noun head (e.g. 20 = block 19 local)
    'gauss_ctx_pv': '',                         # en-pv rows -> deepest global layer (e.g. 22)
}

def cfg_sets(*extra):
    out = []
    for k, v in OVERRIDES.items():
        if v not in (None, ''):
            out += ['--set', f'{k}={v}']
    for e in extra:
        out += ['--set', e]
    return out

# ---- memory ---------------------------------------------------------------
# expandable segments reduce T4 fragmentation; inherited by the subprocess
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ---- HuggingFace push (optional) -----------------------------------------
HF_REPO_ID = os.environ.get('HF_REPO_ID', 'AmnO-O/compartment-weights')
HF_PRIVATE = os.environ.get('HF_PRIVATE', 'true').lower() not in ('0','false','no')

def _hf_token() -> str:
    tok = os.environ.get('HF_TOKEN', '')
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        return ''

# ---- ensure imports exist (Kaggle already ships them) ---------------------
for m in ('torch', 'transformers', 'pandas', 'numpy', 'sklearn', 'scipy', 'yaml'):
    if not _importable(m):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', m], check=False)

def run(args):
    print('\n>>>', ' '.join(args))
    subprocess.run(args, cwd=REPO, check=True)


In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# --- train80: single compound-level 80/20 split ---------------------------
# gauss-only package: python -m src.run  (no warmup / no reg or softmax heads)
if MODE in ('all', 'train80'):
    args = [
        sys.executable, '-m', 'src.run', '--device', 'cuda',
        '--set', 'num_workers=0',            # chống deadlock Kaggle
        '--set', 'batch_size=64',
        '--set', 'freeze_epochs=3',
        '--set', 'head_lr=2e-4',
    ]

    if CONFIG_FILE:
        args += ['--config', CONFIG_FILE]

    args += list(cfg_sets())              # OVERRIDES win over the config file

    if TRAIN_EXTRA:
        args += [x.strip() for x in TRAIN_EXTRA.split(',') if x.strip()]

    run(args)


In [ ]:
# --- Push trained weights to HuggingFace --------------------------------
# Needs only HF_TOKEN (Kaggle Secret or env var) + HF_REPO_ID set above.
# Skipped automatically when HF_REPO_ID is empty.
if HF_REPO_ID and MODE in ('all', 'train80'):
    if not _hf_token():
        print('SKIP push: no HF_TOKEN found (set Kaggle Secret named HF_TOKEN).')
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=_hf_token())
        api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True, repo_type='model')
        work = Path('/kaggle/working')
        # upload all model checkpoints
        models_dir = work / 'models'
        if models_dir.is_dir():
            api.upload_folder(
                folder_path=str(models_dir),
                repo_id=HF_REPO_ID,
                path_in_repo='models',
                allow_patterns='*.pt'
            )
            pt_files = list(models_dir.glob('*.pt'))
            print(f'pushed {len(pt_files)} model file(s)')
        # upload key artifacts
        for name in ('config.json', 'metrics.json', 'history.json'):
            src = work / name
            if src.is_file():
                api.upload_file(path_or_fileobj=str(src), path_in_repo=name, repo_id=HF_REPO_ID)
                print(f'pushed {name}')
        print(f'HF repo: https://huggingface.co/{HF_REPO_ID}')
else:
    print('push skipped (HF_REPO_ID empty or MODE=%s)' % MODE)


In [ ]:
import json

work = Path('/kaggle/working')
p = work / 'metrics.json'
if p.is_file():
    print('--- metrics.json (val rho on holdout 20%) ---')
    print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2))
else:
    print('no metrics.json yet')


### Lưu ý
- Tín hiệu tốt: `val_rho_mean` (rho trung bình Mod/Head trên holdout 20%, split theo compound).
- `gauss_dedicated=1` + `gauss_ctx_mod=19` + `gauss_ctx_head=20` + `gauss_ctx_pv=22` = variant đang test.
- predict / train5 / resume chưa wire trong `src/` (chạy được qua `mm/` cũ nếu cần trial submission).
